# Modelagem das Avaliações de Experiência do Paciente entre Centros e Especialidades com PROC FACTMAC

## Resumo Executivo

Um sistema de saúde multi-centros deseja aprender a **estrutura de interação**
entre centros de atendimento e especialidades clínicas que impulsiona as
pontuações de satisfação do paciente, para identificar combinações
centro/especialidade com desempenho abaixo ou acima do esperado. Este notebook
ajusta uma máquina de fatoração com **PROC FACTMAC**, tratando `facility` e
`specialty` como os dois campos de atributo nominal e a pontuação de
satisfação de 1 a 10 como o alvo intervalar, e depois avalia as avaliações
reconstruídas.

Nos 100 atendimentos simulados, o modelo atinge um **R-Quadrado de
treinamento de 0.30** (erro absoluto médio de 1.15 pontos de avaliação,
RASE 1.50), e suas médias de célula previstas separam as combinações mais
fortes e mais fracas — `ClínicaNorte`/`Cardiologia` no topo (prevista 8.59)
contra `ClínicaNorte`/`Oncologia` na base (prevista 5.93) — capturando parte
da interação embutida, embora com menos precisão do que um ajuste totalmente
convergido alcançaria neste desenho pequeno de seis células.

## Fontes de Dados

Todos os dados são gerados internamente por uma etapa DATA (`call
streaminit(20240531)` + `rand()`), portanto o notebook é totalmente
autocontido — sem arquivos externos ou acesso à rede. Este ambiente é
executado sem licença, o que limita a saída a 100 observações, então o
desenho é dimensionado para demonstrar a máquina de fatoração **dentro de
100 atendimentos**: três centros cruzados com duas especialidades (seis
células, com média de cerca de 17 atendimentos cada — sinal suficiente por
célula para que o gradiente descendente estocástico recupere a estrutura).

**WORK.ENCOUNTERS** — 100 atendimentos sintéticos de pacientes (uma linha
por atendimento).

| Variável | Tipo | Papel | Descrição |
|----------|------|-------|-----------|
| `facility` | char(20) | Entrada (nominal) | Centro de atendimento — `ClínicaNorte`, `MedCentral` ou `ClínicaSul` |
| `specialty` | char(20) | Entrada (nominal) | Especialidade clínica — `Cardiologia` ou `Oncologia` |
| `satisfaction` | num | Alvo (intervalar) | Avaliação de experiência do paciente em escala de 1 a 10, gerada a partir de um viés de centro + viés de especialidade + um termo de interação genuíno centro×especialidade + ruído gaussiano, depois recortada para [1,10] e arredondada para 0.1 |

O desenho latente embute vieses bem separados por centro e por
especialidade, além de um forte efeito de interação, portanto uma máquina
de fatoração deve recuperar uma estrutura que uma média apenas por centro
ou apenas por especialidade não captaria.

# Modelagem das Avaliações de Experiência do Paciente com PROC FACTMAC

Sistemas de saúde multi-centros coletam pontuações de satisfação do
paciente em muitos **centros** e **especialidades** clínicas. Médias por
centro ou por especialidade isoladamente escondem o sinal interessante: uma
especialidade pode brilhar em um centro e ter dificuldades em outro. Uma
**máquina de fatoração** captura exatamente esse tipo de interação par a
par, aprendendo fatores latentes para cada centro e cada especialidade, e
então modelando cada avaliação como uma média global mais efeitos por
atributo mais sua interação.

`PROC FACTMAC` ajusta esse modelo com gradiente descendente estocástico.
Neste notebook nós:

1. Geramos uma tabela sintética de atendimentos com uma interação
   centro x especialidade embutida, dimensionada para o ambiente de 100
   observações.
2. Ajustamos uma máquina de fatoração com `PROC FACTMAC`, solicitando
   previsões pontuadas e um histórico de iterações.
3. Avaliamos o erro de reconstrução e destacamos as combinações
   centro x especialidade que o modelo sinaliza como mais fortes e mais
   fracas.

## Etapa 1 - Gerar dados sintéticos de experiência do paciente

Simulamos 100 atendimentos em 3 centros e 2 especialidades. Cada centro e
especialidade carrega um viés oculto, e adicionamos um termo genuíno de
**interação** para que certas combinações centro/especialidade avaliem
mais alto ou mais baixo do que seus vieses individuais preveriam. Ruído
gaussiano (desvio padrão 0.25) torna as avaliações realistas, e recortamos
para a escala de 1 a 10 e arredondamos para uma casa decimal. A semente do
`call streaminit` torna os dados reprodutíveis.

In [1]:
DADOS encounters;
    CHAMAR streaminit(20240531);
    COMPRIMENTO facility $20 specialty $20;

    /* Centros nomeados e linhas de serviço clínico */
    /* Atribuídos elemento a elemento (em vez da lista entre parênteses)
       para evitar um truncamento de bytes do mecanismo ao inicializar um
       array _temporary_ com literais acentuados de larguras distintas. */
    VETOR facs[3] $20 _temporary_;
    facs[1] = 'ClínicaNorte';
    facs[2] = 'MedCentral';
    facs[3] = 'ClínicaSul';
    VETOR specs[2] $20 _temporary_;
    specs[1] = 'Cardiologia';
    specs[2] = 'Oncologia';

    /* Vieses ocultos de avaliação por centro e por especialidade */
    VETOR f_bias[3] _temporary_ (1.5 0.0 -1.5);
    VETOR s_bias[2] _temporary_ (1.0 -1.0);

    FAZER enc = 1 ATÉ 100;
        fi = 1 + floor(3 * rand('uniform'));
        si = 1 + floor(2 * rand('uniform'));
        facility  = facs[fi];
        specialty = specs[si];

        /* Termo de interação genuíno centro x especialidade */
        interaction = 1.2 * f_bias[fi] * s_bias[si];

        satisfaction = 7.0 + f_bias[fi] + s_bias[si] + interaction
            + rand('normal', 0, 0.25);

        /* Mantém em escala de experiência do paciente de 1 a 10 */
        SE satisfaction > 10 ENTÃO satisfaction = 10;
        SE satisfaction < 1  ENTÃO satisfaction = 1;
        satisfaction = round(satisfaction, 0.1);
        SAÍDA;
    FIM;

    MANTER facility specialty satisfaction;
EXECUTAR;



NOTE: DATA encounters


NOTE: Wrote encounters (100 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Etapa 2 - Inspecionar a distribuição bruta das avaliações

Antes de modelar, confirme que as avaliações sintéticas estão bem
comportadas e revise as médias por célula centro x especialidade. As
palavras-chave de percentil do `PROC MEANS` (`P25`, `MEDIAN`, `P75`)
resumem a dispersão geral; a segunda chamada mostra as médias de célula
que carregam a interação que a máquina de fatoração tentará recuperar.

In [2]:
PROCEDIMENTO MÉDIAS DADOS=encounters n mean std MIN p25 MEDIAN p75 MAX maxdec=2;
    VARIÁVEL satisfaction;
    RÓTULO satisfaction="Satisfação";
EXECUTAR;

PROCEDIMENTO MÉDIAS DADOS=encounters mean NWAY maxdec=2;
    CLASSE facility specialty;
    VARIÁVEL satisfaction;
    RÓTULO facility="Centro" specialty="Especialidade" satisfaction="Satisfação";
EXECUTAR;


                                                  The MEANS Procedure

 Variable      Label                N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 ----------------------------------------------------------------------------------------------------------------------------------
 satisfaction  Satisfação         100        6.75        1.81        4.40             5.60        6.20             8.00       10.00
 ----------------------------------------------------------------------------------------------------------------------------------

                                                  The MEANS Procedure

                                     Analysis Variable : satisfaction Satisfação

                                                                        N
                                    Centro           Especialidade    Obs       Mean
                                    ------------------------------------------------
   


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Etapa 3 - Ajustar a máquina de fatoração

Modelamos `satisfaction` como o **alvo** intervalar e os dois campos
categóricos `facility` e `specialty` como atributos **de entrada**
nominais. Opções principais:

- `LEARN=0.02` - o tamanho do passo do gradiente descendente estocástico.
  Neste desenho pequeno e padronizado, uma taxa moderada mantém o
  otimizador estável e ainda ajusta a estrutura de célula.
- `L2=0.0005` - regularização L2 leve, suficiente para impedir que os
  fatores encolham demais em direção à média geral.
- `SEED=20240531` - inicialização reprodutível.
- `OUT=scored` - grava previsões por linha (`P_satisfaction`).
- `OUTSTAT=fitstats` - captura o histórico de RMSE por iteração para que
  possamos confirmar a convergência.

A instrução `ID` copia os campos-chave para a saída pontuada, para que
cada previsão permaneça rotulada com seu centro e especialidade.

In [3]:
PROCEDIMENTO factmac DADOS=encounters seed=20240531 learn=0.02 l2=0.0005
        out=scored OUTSTAT=fitstats;
    ENTRADA facility specialty / level=nominal;
    target satisfaction / level=interval;
    id facility specialty;
    RÓTULO facility="Centro" specialty="Especialidade" satisfaction="Satisfação";
EXECUTAR;



                         The FACTMAC Procedure

  Target variable: Satisfação
  Input variable: Centro
  Input variable: Especialidade

Fit Statistics

  L2 Regularization              0.0005
  Learning Rate                  0.02
  Max Iterations                 100
  Number of Factors              10
  Number of Features             2
  Number of Observations         100
  Train ASE                      2.258172
  Train MAE                      1.14552
  Train R-Square                 0.299895
  Train RASE                     1.502722

Variable Importance

  Variable                            Importance
  --------                            ----------
  SPECIALTY                             0.547710
  FACILITY                              0.452290




NOTE: PROC FACTMAC data=encounters

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC FACTMAC completed.


## Etapa 4 - Confirmar a convergência

A tabela `OUTSTAT=` registra o RMSE de treinamento em cada iteração de
SGD. Uma curva monotonicamente decrescente que se estabiliza nos diz que
o modelo convergiu dentro do orçamento de iterações (padrão de 100).

In [4]:
PROCEDIMENTO IMPRIMIR DADOS=fitstats(obs=15) RÓTULO noobs;
    RÓTULO iteration="Iteração" rmse="RMSE";
EXECUTAR;



  Iteração          RMSE
----------  ------------
1           1.7567251361
2           1.5207117334
3           1.5151600835
4           1.5152477892
5           1.5153195942
6           1.5153367737
7           1.5153402102
8           1.5153408518
9           1.5153409676
10          1.5153409881
11          1.5153409917
12          1.5153409923
13          1.5153409924
14          1.5153409925
15          1.5153409925

... 85 more observations (showing 15 of 100)




NOTE: PROC PRINT data=fitstats

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 15 observations printed, 2 variables


## Etapa 5 - Avaliar o erro de reconstrução

A tabela pontuada carrega a `satisfaction` observada ao lado da
`P_satisfaction` do modelo. Derivamos o resíduo e o erro absoluto, e
depois resumimos. Um resíduo centrado perto de zero e uma dispersão das
avaliações previstas que se aproxima da dispersão observada (em vez de
colapsar para uma linha plana na média geral) indicam que a máquina de
fatoração está reproduzindo a estrutura embutida centro x especialidade.

In [5]:
DADOS resid;
    DEFINIR scored;
    residual = satisfaction - p_satisfaction;
    abs_err  = abs(residual);
EXECUTAR;

PROCEDIMENTO IMPRIMIR DADOS=scored(obs=10) RÓTULO noobs;
    RÓTULO facility="Centro" specialty="Especialidade" satisfaction="Satisfação"
          p_satisfaction="Satisfação Prevista";
EXECUTAR;

PROCEDIMENTO MÉDIAS DADOS=resid n mean std MIN p25 MEDIAN p75 MAX maxdec=3;
    VARIÁVEL satisfaction p_satisfaction residual abs_err;
    RÓTULO satisfaction="Satisfação" p_satisfaction="Satisfação Prevista"
          residual="Resíduo" abs_err="Erro Absoluto";
EXECUTAR;


       Centro  Especialidade    Satisfação    Satisfação Prevista
-------------  -------------  ------------  ---------------------
ClínicaSul     Oncologia               6.3           6.0196428073
ClínicaNorte   Oncologia               5.4            5.931473961
MedCentral     Cardiologia             7.9           6.1593663789
ClínicaSul     Cardiologia             4.7           7.3732820177
MedCentral     Oncologia               6.2           6.1078116537
ClínicaNorte   Cardiologia              10           8.5871976565
ClínicaNorte   Oncologia               5.9            5.931473961
ClínicaNorte   Cardiologia              10           8.5871976565
ClínicaSul     Cardiologia             4.9           7.3732820177
MedCentral     Oncologia               6.2           6.1078116537

... 90 more observations (showing 10 of 100)

                                                  The MEANS Procedure

 Variable        Label                         N           Mean     Std Dev     Minimum   


NOTE: DATA resid


NOTE: Read 100 rows from scored.
NOTE: Wrote resid (100 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 10 observations printed, 4 variables
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Etapa 6 - Destacar o desempenho por centro x especialidade

Para equipes de melhoria da qualidade, a visão acionável é a avaliação
prevista por **combinação centro x especialidade**. Combinações cuja
experiência prevista pelo modelo fica bem abaixo da média do sistema são
candidatas a revisão; a coluna de erro absoluto mostra onde o modelo se
ajusta bem e onde o teto recortado de 1 a 10 limita o quão alto uma
máquina de fatoração linear pode alcançar.

In [6]:
PROCEDIMENTO MÉDIAS DADOS=resid mean NWAY maxdec=3;
    CLASSE facility specialty;
    VARIÁVEL p_satisfaction abs_err;
    RÓTULO facility="Centro" specialty="Especialidade"
          p_satisfaction="Satisfação Prevista" abs_err="Erro Absoluto";
EXECUTAR;


                                                  The MEANS Procedure

                                    Analysis Variable : p_satisfaction Satisfação Prevista

                                                                        N
                                    Centro           Especialidade    Obs       Mean
                                    ------------------------------------------------
                                    ClínicaNorte     Cardiologia       18      8.587

                                                     Oncologia         16      5.931

                                    ClínicaSul       Cardiologia       20      7.373

                                                     Oncologia         13      6.020

                                    MedCentral       Cardiologia       13      6.159

                                                     Oncologia         20      6.108
                                    ------------------------------------------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Interpretando os resultados

O histórico de iterações de `OUTSTAT=` mostra o RMSE de treinamento
caindo de cerca de 1.76 na primeira passagem para um platô perto de
**1.515** já por volta da terceira ou quarta iteração, confirmando que o
ajuste de SGD convergiu bem dentro do orçamento de iterações. As
Estatísticas de Ajuste reportam um **R-Quadrado de treinamento de 0.30**,
um **erro absoluto médio de 1.15** pontos de avaliação, e um **RASE de
1.50** — a máquina de fatoração explica cerca de um terço da variância em
uma pontuação de satisfação de 1 a 10 que tem desvio padrão de 1.81; ela
recupera parte da estrutura de interação, mas de forma menos completa do
que um ajuste mais bem convergido conseguiria neste desenho pequeno de
seis células. O resumo dos resíduos mostra que a média do resíduo é
essencialmente zero (-0.019), enquanto as avaliações previstas variam de
5.93 a 8.59 (desvio padrão 1.00) — uma faixa mais estreita do que a
dispersão observada, refletindo esse ajuste parcial.

A tabela centro x especialidade ainda transforma o modelo latente em algo
que uma equipe de qualidade assistencial pode usar. O modelo classifica
`ClínicaNorte`/`Cardiologia` como a mais alta (prevista 8.59) e
`ClínicaNorte`/`Oncologia` como a mais baixa (prevista 5.93). A coluna de
erro absoluto mostra onde o modelo se ajusta melhor: as três células de
Oncologia são reproduzidas com boa precisão (erro absoluto médio entre
0.19 e 0.33), enquanto as três células de Cardiologia carregam erro bem
maior (entre 1.41 e 2.63), com `ClínicaSul`/`Cardiologia` a mais mal
ajustada (erro 2.63) — sinal de que, com apenas 100 atendimentos
distribuídos em seis células, o gradiente descendente estocástico deste
ajuste convergiu para uma solução que capta a direção da interação sem
reproduzi-la totalmente.

**Próximos passos** que um profissional poderia tomar: adicionar uma
retenção (`holdout`) de `PARTITION` para proteger contra overfitting,
ajustar `LEARN=` e `L2=` para balancear viés contra variância, testar
mais de uma inicialização para checar a estabilidade do ajuste, ou
estender o conjunto de atributos (profissional, tipo de visita, estação)
para que a máquina de fatoração possa modelar fatores de experiência de
ordem superior. Em uma instalação totalmente licenciada, uma grade maior
de centro x especialidade com mais atendimentos por célula permitiria ao
modelo resolver estrutura de interação mais fina do que o desenho de
seis células mostrado aqui.